导入必要的模块

In [ ]:
#!/usr/bin/python3
#coding=utf8
import sys
sys.path.append('d:/robot/daran')
import cv2
import copy
import time
import math
import camera
from cv_ImgAddText import *
from lab_config import color_range
import threading
import numpy as np
import time
import math
import gripper
import DrEmpower_can as dr
import arm_robot as robot
import keyboard  # 替代方案

相机的机械臂初始化

In [ ]:

#设置并打印摄像头分辨率
size = (480, 360)
MyCamera = camera.USBCamera(size)
print('Frame size: ' + str(size[0]) + 'X' + str(size[1]) + '\n')

##初始化机械臂
l_p = 0 # 工具参考点到电机输出轴表面的距离，单位mm（所有尺寸参数皆为mm）
l_p_mass_center = 0 # 工具（负载）质心到 6 号关节输出面的距离
G_p = 0 # 负载重量，单位kg，所有重量单位皆为kg
max_list_temp = [85, 215, 149, 142, 179, 179]  # 关节模型角度最大值,1号关节目的是保护线缆，并且到达工作空间边缘；2号关节到达工作空间边缘；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达
min_list_temp = [-85, -15, -149, -142, -179, -179]  # 关节模型角度最小值，1号关节目的是不装到装在桌边的竖杆（安装摄像头）；2号关节目的是在伸直的时候不打到桌子；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达

# # 机械臂对象初始化函数函数
ro = robot.arm_robot(L_p=l_p, L_p_mass_center=l_p_mass_center, MAX_list_temp=max_list_temp, MIN_list_temp=min_list_temp, G_p=G_p)

# 全局变量
# 边界
x_min, x_max = 100, 320
y_min, y_max = -10, 190
z_min, z_max = 80, 150
# 初始位置
x = 200
y = 58
z = 140

#初始化到工作空间
ro.set_arm_pose(pl_temp=[x,y, z], theta_P_R_Y=[0, 90, 0], speed=2, param=10, mode=1)

运动到指定位置

In [ ]:
# 运动到指定位置
x = 100
y = 58
z = 140
# 边界检查
x = max(x_min, min(x_max, x))
y = max(y_min, min(y_max, y))
z = max(z_min, min(z_max, z))
# 运动到指定位置
ro.set_arm_pose(pl_temp=[x,y, z], theta_P_R_Y=[0, 90, 0], speed=2, param=10, mode=1)

计算到指定坐标时，回转旋转的角度，用于修正Y，使手抓始终水平

In [ ]:
# 初始位置
x = 190
y = 180
z = 140
# 逆运动学，算出各关节角度
ro.inverse_kinematics(pl_temp=[x,y, z], theta_P_R_Y=[0, 90, 0], ud=0)
# 获取当前关节角度（弧度）
theta_rad = ro.theta
# 转换为角度
theta_deg = [math.degrees(rad) for rad in theta_rad]
# 让姿态角Y等于回转角度可以使手抓始终平等于y轴
# 运动到指定位置
ro.set_arm_pose(pl_temp=[x,y, z], theta_P_R_Y=[0, 90, theta_deg[0]], speed=2, param=10, mode=1)

松开手抓

In [ ]:
gripper.grasp(wideth=60, speed=10, force=120)

合拢手抓

In [ ]:
gripper.grasp(wideth=5, speed=10, force=80)

查看当前关节模型角度

In [ ]:
# 查看当前关节模型角度函数(通过回读一体化关节角度计算)*******************************************
ro.detect_joints()

查看当前位姿函数

In [ ]:
# 查看当前位姿函数(通过回读一体化关节角度计算)*******************************************
ro.detect_pose()

角关节驱动机械臂

In [ ]:
joint_angle = [0,0,0,0,0,0]
ro.set_arm_joints(joint_angle)

In [ ]:
joint_angle = [0,90,-45,0,0,0]
ro.set_arm_joints(joint_angle)

通过键盘移动机械臂，确定机械臂的坐标与移动范围

In [ ]:
#!/usr/bin/python3
#coding=utf8
import sys
sys.path.append('d:/robot/daran')
import cv2
import copy
import time
import math
import camera
from cv_ImgAddText import *
from lab_config import color_range
import threading
import numpy as np
import time
import math
import gripper
import DrEmpower_can as dr
import arm_robot as robot
import keyboard  # 替代方案


#设置并打印摄像头分辨率
size = (480, 360)
MyCamera = camera.USBCamera(size)
print('Frame size: ' + str(size[0]) + 'X' + str(size[1]) + '\n')

##初始化机械臂
l_p = 0 # 工具参考点到电机输出轴表面的距离，单位mm（所有尺寸参数皆为mm）
l_p_mass_center = 0 # 工具（负载）质心到 6 号关节输出面的距离
G_p = 0 # 负载重量，单位kg，所有重量单位皆为kg
max_list_temp = [85, 215, 149, 142, 179, 179]  # 关节模型角度最大值,1号关节目的是保护线缆，并且到达工作空间边缘；2号关节到达工作空间边缘；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达
min_list_temp = [-85, -15, -149, -142, -179, -179]  # 关节模型角度最小值，1号关节目的是不装到装在桌边的竖杆（安装摄像头）；2号关节目的是在伸直的时候不打到桌子；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达

# # 机械臂对象初始化函数函数
ro = robot.arm_robot(L_p=l_p, L_p_mass_center=l_p_mass_center, MAX_list_temp=max_list_temp, MIN_list_temp=min_list_temp, G_p=G_p)

# ########轨迹运动之前可先调整一下 pid，以获得更好的曲线平滑度
ro.set_pid(joint_num=1, P=10, I=8, D=0.75)
ro.set_pid(joint_num=2, P=25, I=9, D=0.95)
ro.set_pid(joint_num=3, P=25, I=9, D=0.95)
ro.set_pid(joint_num=4, P=10, I=9, D=3.2)
ro.set_pid(joint_num=5, P=12, I=5, D=0.1)
ro.set_pid(joint_num=6, P=12, I=5, D=0.096)

# 全局变量
# 边界
x_min, x_max = 100, 320
y_min, y_max = -10, 190
z_min, z_max = 80, 150
# 初始位置
x = 200
y = 58
z = 140

#初始化到工作空间
ro.set_arm_pose(pl_temp=[x,y, z], theta_P_R_Y=[0, 90, 0], speed=2, param=10, mode=1)
# 合拢手抓
gripper.grasp(wideth=5, speed=10, force=80)

# 线程锁
position_lock = threading.Lock()

# 定义选框坐标变量
drag_start = None
def onMouse(event, x, y, flags, param):
    global drag_start
    if event == cv2.EVENT_LBUTTONUP:
        drag_start = (x, y)  # 记录拖拽起始点
    

def check_keys():
    global x, y, z
    with position_lock:
        if keyboard.is_pressed('up'):
            x += 1
        elif keyboard.is_pressed('down'):
            x -= 1
        elif keyboard.is_pressed('left'):
            y += 1
        elif keyboard.is_pressed('right'):
            y -= 1
        elif keyboard.is_pressed('page up'):
            z += 1
        elif keyboard.is_pressed('page down'):
            z -= 1

    # 下次检测
    threading.Timer(0.1, check_keys).start()

# 启动按键检测线程
check_keys()
# 先创建窗口，再绑定事件
cv2.imshow("image", np.zeros((size[1], size[0], 3), dtype=np.uint8))  # 创建空白窗口
cv2.setMouseCallback("image", onMouse)  # ✅ 此时窗口已存在

while True:      
    orgFrame = MyCamera.getframe()
    if orgFrame is not None:
        # 获取当前位置（线程安全）
        with position_lock:
            current_x, current_y, current_z = x, y, z

        # 边界检查
        current_x = max(x_min, min(x_max, current_x))
        current_y = max(y_min, min(y_max, current_y))
        current_z = max(z_min, min(z_max, current_z))

        # 更新机械臂位置
        ro.set_arm_pose(pl_temp=[current_x, current_y, current_z],
                        theta_P_R_Y=[0, 90, 0], speed=2, param=10, mode=1) 

        if(drag_start): 
            cv2.putText(orgFrame, '(' + str(drag_start[0]) + ',' + str(drag_start[1]) + ')', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.putText(orgFrame, '(' + str(x)+ ',' + str(y) + ',' + str(z) + ')', (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.5,  (0, 0, 255), 1)     
        cv2.imshow("image", orgFrame)
        key = cv2.waitKey(1)
        if key == 27 :
            break
    
MyCamera.shutdown()        

画正方形

In [ ]:

#!/usr/bin/python3
#coding=utf8
import sys
sys.path.append('d:/robot/daran')
import cv2
import copy
import time
import math
import camera
from cv_ImgAddText import *
from lab_config import color_range
import threading
import numpy as np
import time
import math
import gripper
import DrEmpower_can as dr
import arm_robot as robot
import keyboard  # 替代方案


#设置并打印摄像头分辨率
size = (480, 360)
MyCamera = camera.USBCamera(size)
print('Frame size: ' + str(size[0]) + 'X' + str(size[1]) + '\n')

##初始化机械臂
l_p = 0 # 工具参考点到电机输出轴表面的距离，单位mm（所有尺寸参数皆为mm）
l_p_mass_center = 0 # 工具（负载）质心到 6 号关节输出面的距离
G_p = 0 # 负载重量，单位kg，所有重量单位皆为kg
max_list_temp = [85, 215, 149, 142, 179, 179]  # 关节模型角度最大值,1号关节目的是保护线缆，并且到达工作空间边缘；2号关节到达工作空间边缘；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达
min_list_temp = [-85, -15, -149, -142, -179, -179]  # 关节模型角度最小值，1号关节目的是不装到装在桌边的竖杆（安装摄像头）；2号关节目的是在伸直的时候不打到桌子；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达

# # 机械臂对象初始化函数函数
ro = robot.arm_robot(L_p=l_p, L_p_mass_center=l_p_mass_center, MAX_list_temp=max_list_temp, MIN_list_temp=min_list_temp, G_p=G_p)

# 全局变量
# 边界
x_min, x_max = 100, 320
y_min, y_max = -10, 190
z_min, z_max = 80, 150
# 初始位置
x = 200
y = 58
z = 140

#初始化到工作空间
ro.set_arm_pose(pl_temp=[x,y, z], theta_P_R_Y=[0, 90, 0], speed=2, param=10, mode=1)
# 合拢手抓
gripper.grasp(wideth=5, speed=10, force=80)


####轨迹跟踪函数***********************************************
############画正方形#################
def draw_rectangle(pl=[200, 58, 140], l=30, h=30):
    ''''在水平面上画正方形
    pl: 长方形左上角坐标（起始点），其中pl[2]代表作图平面与全局坐标系z轴的焦点的z坐标
    l: 宽度
    h: 高度
    '''
    n= 50 # 每条边分割的点数（数量越多画得越慢）
    l_delta = l/n
    h_delta = h/n
    pl_list = []
    pl_list.append(pl)
    l1 = pl[1]
    for i in range(1, n+1):
        pl_temp = [pl[0], pl[1]-i*l_delta, pl[2]]
        pl_list.append(pl_temp)
    print(pl_temp)
    for i in range(1, n+1):
        pl_temp1 = [pl_temp[0]-i*h_delta, pl_temp[1], pl_temp[2]]
        pl_list.append(pl_temp1)
    print(pl_temp1)
    for i in range(1, n+1):
        pl_temp2 = [pl_temp1[0], pl_temp1[1]+i*l_delta, pl_temp1[2]]
        pl_list.append(pl_temp2)
    print(pl_temp2)
    for i in range(1, n+1):
        pl_temp3 = [pl_temp2[0]+i*h_delta, pl_temp2[1], pl_temp2[2]]
        pl_list.append(pl_temp3)
    print(pl_temp3)
    print(pl_list)
    return pl_list

def get_joint_angles(pl_list, theta_P_R_Y=[0, 90, 0]):
    """
    根据位置列表 pl_list，求取对应的关节角度列表（角度制）

    参数:
        pl_list (list): 位置点列表，如 [[x1, y1, z1], [x2, y2, z2], ...]
        theta_P_R_Y (list): 姿态角（默认为 [0, 90, 0]）

    返回:
        list: 每个位置点对应的关节角度列表（角度制）
    """
    joint_angles_list = []
    for point in pl_list:
        # 调用逆运动学函数
        ro.inverse_kinematics(pl_temp=point, theta_P_R_Y=theta_P_R_Y, ud=0)
        # 获取当前关节角度（弧度）并转换为角度
        theta_rad = ro.theta
        theta_deg = [math.degrees(rad) for rad in theta_rad]
        joint_angles_list.append([0,90,theta_deg[0]])
    return joint_angles_list

def check_boundaries(pl_list, x_min, x_max, y_min, y_max, z_min, z_max):
    """
    检查并裁剪 pl_list 中的坐标点，确保不超出边界。

    参数:
        pl_list (list): 原始坐标列表，如 [[x1, y1, z1], [x2, y2, z2], ...]
        x_min, x_max (float): x 轴边界
        y_min, y_max (float): y 轴边界
        z_min, z_max (float): z 轴边界

    返回:
        list: 裁剪后的安全坐标列表
    """
    safe_pl_list = []
    for point in pl_list:
        x, y, z = point
        # 裁剪到边界
        x_clipped = max(x_min, min(x_max, x))
        y_clipped = max(y_min, min(y_max, y))
        z_clipped = max(z_min, min(z_max, z))
        # 记录越界警告（可选）
        if x_clipped != x:
            print(f"警告: 点 {point} 的 x 超出边界，已裁剪至 {x_clipped}")
        if y_clipped != y:
            print(f"警告: 点 {point} 的 y 超出边界，已裁剪至 {y_clipped}")
        if z_clipped != z:
            print(f"警告: 点 {point} 的 z 超出边界，已裁剪至 {z_clipped}")
        safe_pl_list.append([x_clipped, y_clipped, z_clipped])
    return safe_pl_list
#
#
pl_list = draw_rectangle(pl=[200, 58, 140], l=50, h=50) #
# 应用边界检查
safe_pl_list = check_boundaries(
    pl_list=pl_list,
    x_min = x_min, x_max = x_max,
    y_min = y_min, y_max = y_max,
    z_min = z_min, z_max = z_max
)
# 获取对应的关节角度列表
joint_angles = get_joint_angles(safe_pl_list)

# ########轨迹运动之前可先调整一下 pid，以获得更好的曲线平滑度
ro.set_pid(joint_num=1, P=10, I=8, D=0.75)
ro.set_pid(joint_num=2, P=25, I=9, D=0.95)
ro.set_pid(joint_num=3, P=25, I=9, D=0.95)
ro.set_pid(joint_num=4, P=10, I=9, D=3.2)
ro.set_pid(joint_num=5, P=12, I=5, D=0.1)
ro.set_pid(joint_num=6, P=12, I=5, D=0.096)


# ########控制机械臂末端连续运动到多个指定位置和姿态函数(必须单独一次性使用)
# # ro.set_arm_poses(pls_temp=pl_list, theta_P_R_Ys_temp=[[0, 90, 0]], t=10) # 控制机械臂末端连续运动到多个指定位置和姿态函数(必须单独一次性使用)
ro.set_arm_poses_curve_pre(pls_temp=safe_pl_list, theta_P_R_Ys_temp=joint_angles) # 预设机械臂末端轨迹函数
ro.set_arm_poses_curve_start_point(2) # 运动到轨迹起始位置函数
while True:
    ro.set_arm_poses_curve_do(5) # 末端轨迹执行函数，参数为大致运行时间